# TODO: You must set checkpoint_directory to where you want to save and load the model

In [19]:
checkpoint_directory ="/Users/jameswalker/Programming/HazardousCheckpoint1"
# checkpoint_directory = "/content/drive/MyDrive/rllib_checkpoints"

In [15]:
import ray 

ray.shutdown()
ray.init() 

2025-04-25 19:03:11,853	INFO worker.py:1852 -- Started a local Ray instance.


Python version:,3.9.17
Ray version:,2.44.1


In [16]:
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from tqdm import tqdm 
from metadrive.envs.hazardous_metadrive_env import HazardousMetaDriveEnv

# Not sure if sgd_minibatch_size is being handled properly, but train_batch_size and worker_num are for sure
# worker_num = 1
# train_batch_size = int(1024)
# sgd_minibatch_size = max(256, int(train_batch_size/10))

# https://docs.ray.io/en/latest/rllib/package_ref/doc/ray.rllib.algorithms.algorithm_config.AlgorithmConfig.html
config = ( PPOConfig()
    .environment(
        HazardousMetaDriveEnv,
        disable_env_checking=True # TODO Is this right?
        )
    .env_runners(
        num_env_runners=3, 
        num_cpus_per_env_runner=1,
        num_gpus_per_env_runner=0
        )
    .framework('torch')
    # The below .learners comes from the migration guide when they talk about no GPU but multi-CPU
    .learners(
        num_learners=2,
        num_cpus_per_learner=2,
        # num_gpus_per_learner=0,
    )
    .training(
        # train_batch_size=train_batch_size, # Deprecated
        train_batch_size_per_learner=int(2048), # Replacement for train_batch_size
        # sgd_minibatch_size=sgd_minibatch_size, # This got deprecated
        minibatch_size=int(256), # This might be the replacement
        lr=1e-4,
        entropy_coeff=0.001,
    )    
    # .rl_module will be necessary for defining custom multi-agent models
    .rl_module(
        # Use a non-default 32,32-stack with ReLU activations.
        # This is for PPO since it uses multi-layer perceptrons (MLP)
        model_config=DefaultModelConfig(
            fcnet_hiddens=[32, 32],
            fcnet_activation="relu",
        )
    )
    # The below, commented .training illustrates grid search for learning rate,
    # comes from https://github.com/ray-project/ray/blob/master/rllib/examples/gpus/fractional_gpus_per_learner.py
    # .training(
    #     lr=tune.grid_search([0.005, 0.003, 0.001, 0.0001])
    # )
    .evaluation(
        # TODO evaluation_interval?
        evaluation_interval=1,
        evaluation_num_env_runners=1, 
        # evaluation_duration=10000,
        # WARNING algorithm_config.py:4593 -- You have specified 1 evaluation workers, but your `evaluation_interval` is 0 or None! Therefore, evaluation doesn't occur automatically with each call to `Algorithm.train()`. Instead, you have to call `Algorithm.evaluate()` manually in order to trigger an evaluation run.
    )
    # .resources is only for setting num_cpus_for_main_process, I don't think we need to mess with it
    # .resources(
    #     num_gpus=int(0),  # This is deprecated
    #     num_cpus_for_main_process=int(1), # Defaults to 1
    # )  
    .debugging(log_level='INFO') # INFO, DEBUG, ERROR, WARN
)



In [17]:
# config.build() is deprecated, use build_algo() instead
algo = config.build_algo()
# print(f"Type of algo: {type(algo)}")


(SingleAgentEnvRunner pid=10063) [INFO] Environment: HazardousMetaDriveEnv
(SingleAgentEnvRunner pid=10063) [INFO] MetaDrive version: 0.4.3
(SingleAgentEnvRunner pid=10063) [INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
(SingleAgentEnvRunner pid=10063) [INFO] Render Mode: none
(SingleAgentEnvRunner pid=10063) [INFO] Horizon (Max steps per agent): 1000
(SingleAgentEnvRunner pid=10063) 2025-04-25 19:03:19,999	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/gymnasium/envs/registration.py:642: UserWarning: WARN: Overriding environment rllib-single-agent-env-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
[IN

(_WrappedExecutable pid=10070) 2025-04-25 19:03:25,237	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!


# TRAIN THE ALGORITHM
100 training iterations took about 2 minutes on my Mac with M1 chip
If setting it up on a server, just try 100 iterations and see if you can save and load the Algorithm instance. After that try 10,000 iterations

In [18]:
import numpy as np
# iteration_num = 10_000
iteration_num = 20
# iteration_num = 1
ENV_RUNNER_RESULTS = "env_runners"
EPISODE_RETURN_MEAN = "episode_return_mean"
EVALUATION_RESULTS = "evaluation"
for iteration in tqdm(range(iteration_num)):
    results = algo.train()
    if ENV_RUNNER_RESULTS in results:
        mean_return = results[ENV_RUNNER_RESULTS].get(EPISODE_RETURN_MEAN, np.nan)
        print(f"iter={iteration+1} R={mean_return}", end="")
        if (EVALUATION_RESULTS in results and ENV_RUNNER_RESULTS in results[EVALUATION_RESULTS]):
            Reval = results[EVALUATION_RESULTS][ENV_RUNNER_RESULTS][EPISODE_RETURN_MEAN]
            print(f" R(eval)={Reval}", end="")
        print()
    # For seeing all the metrics we can use in the result dict():
    # for key in result['env_runners'].keys():
    #     print(f"Key {key}: {result['env_runners'][key]}")
    # # TODO Are these good metrics below? Not sure exactly what either means
    # if iteration % (iteration_num/10) == 0:
        # print(f"Iteration {iteration} Episode Length Mean: {result['env_runners']['episode_len_mean']}")
        # print(f"Iteration {iteration} Agent Episode Returns Mean: {result['env_runners']['agent_episode_returns_mean']}")


  0%|          | 0/20 [00:00<?, ?it/s](SingleAgentEnvRunner pid=10063) [INFO] Assets version: 0.4.3
(_WrappedExecutable pid=10067) 2025-04-25 19:03:25,237	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
(SingleAgentEnvRunner pid=10063) [INFO] Known Pipes: CocoaGraphicsPipe
(SingleAgentEnvRunner pid=10063) [INFO] Start Scenario Index: 0, Num Scenarios : 100
(SingleAgentEnvRunner pid=10064) [INFO] Assets version: 0.4.3 [repeated 3x across cluster]
(SingleAgentEnvRunner pid=10066) [INFO] Known Pipes: CocoaGraphicsPipe [repeated 2x across cluster]
(SingleAgentEnvRunner pid=10066) [INFO] Start Scenario Index: 0, Num Scenarios : 100 [repeated 2x across cluster]
  5%|▌         | 1/20 [00:29<09:17, 29.33s/it]

iter=1 R=407.99044723814933 R(eval)=505.3810822280599


 10%|█         | 2/20 [01:01<09:19, 31.09s/it]

iter=2 R=389.5261864954365 R(eval)=496.4817904662826


 15%|█▌        | 3/20 [01:29<08:21, 29.47s/it]

...

iter=18 R=446.0044104025782 R(eval)=454.8470223183765


 95%|█████████▌| 19/20 [09:18<00:28, 28.29s/it]

iter=19 R=464.1612288164028 R(eval)=457.07653030130945


100%|██████████| 20/20 [09:44<00:00, 29.21s/it]

iter=20 R=441.1960764128015 R(eval)=446.42338516543066


# Save Algorithm instance
"An Algorithm instance typically holds a neural network for computing actions, called the policy, the RL environment you want to optimize against, a loss function, an optimizer, and some code describing the algorithm’s execution logic, like determining when to collect samples, when to update your model, etc."

We can checkpoint an Algorithm to save its state then load it into a new Algorithm instance at a later date.

In [20]:
# Save Algorithm checkpoint.
algo.save_to_path(checkpoint_directory)

# Display the Algorithm RLModule to check we loaded the same RLModule later on
print(type(algo))
algo.get_module()

<class 'ray.rllib.algorithms.ppo.ppo.PPO'>


DefaultPPOTorchRLModule(
  (encoder): TorchActorCriticEncoder(
    (encoder): TorchMLPEncoder(
      (net): TorchMLP(
        (mlp): Sequential(
          (0): Linear(in_features=259, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=32, bias=True)
          (3): ReLU()
        )
      )
    )
  )
  (pi): TorchMLPHead(
    (net): TorchMLP(
      (mlp): Sequential(
        (0): Linear(in_features=32, out_features=4, bias=True)
      )
    )
  )
)

# Loading the Algorithm instance

In [21]:
from ray.rllib.core.rl_module.rl_module import RLModule
from ray.rllib.algorithms.algorithm import Algorithm
import os
from ray.rllib.core import DEFAULT_MODULE_ID
rl_module = RLModule.from_checkpoint(
        os.path.join(
            checkpoint_directory,
            "learner_group",
            "learner",
            "rl_module",
            DEFAULT_MODULE_ID,
        )
    )

In [ ]:
# from ray.rllib.core.rl_module.rl_module import RLModule
# from ray.rllib.algorithms.algorithm import Algorithm
# import os
# from ray.rllib.core import DEFAULT_MODULE_ID

# algo_loaded = Algorithm.from_checkpoint(checkpoint_directory)

# # Compare the Algorithm we loaded with the one we stored earlier
# print(type(algo_loaded))
# algo_loaded.get_module()

(autoscaler +5h35m43s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.
(autoscaler +5h36m18s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


KeyboardInterrupt: 

(autoscaler +5h36m53s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.
(autoscaler +5h37m28s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


https://github.com/ray-project/ray/tree/master/rllib/examples/inference

In [22]:
print(type(algo_loaded.get_module()))

NameError: name 'algo_loaded' is not defined

# How to debug AssertionError
AssertionError: Can not call this API after engine initialization!

Run the cell below. It should fix this error. This error gets raised if you press 'ESC' and shutdown the Environment but the renderer doesn't properly shutdown.

In [34]:
env.close()

2025-04-14 22:48:18.170 python[56773:9790512] IMKClient Stall detected, *please Report* your user scenario attaching a spindump (or sysdiagnose) that captures the problem - (imkxpc_attributesForCharacterIndex:reply:) block performed very slowly (93.88 secs).
2025-04-14 22:48:18.170 python[56773:9790512] IMKClient Stall detected, *please Report* your user scenario attaching a spindump (or sysdiagnose) that captures the problem - (imkxpc_inputSessionDoneSleepWithReply:) block performed very slowly (87.87 secs).


# Testing your trained model
This runs for 300 steps. My model was super bad so it never got all the checkpoints, so I'm not sure what happens if all checkpoints are collected.

In [ ]:
from metadrive.envs.hazardous_metadrive_env import HazardousMetaDriveEnv
from metadrive.envs.safe_metadrive_env import SafeMetaDriveEnv
from metadrive.component.pgblock.first_block import FirstPGBlock
import torch

env = HazardousMetaDriveEnv({
        # "force_render_fps": None,
        # "out_of_route_done":True, # I don't think this changed anything
        # "accident_prob": 0.4,
        # "need_inverse_traffic": True,
        # "map": 4,
        # "use_mesh_terrain": True,
        # "traffic_mode": "trigger",
        # "traffic_density": 0.1, # 0.15-0.2 is good for training
        # "out_of_route_done":True, # We don't want them cutting corners
        # "on_continuous_line_done":True, # Good practice probably
        # "agent_policy": IDMPolicy,
        # "horizon": 100,
        # 'num_scenarios': 100,
        # "start_seed": 129,
        "use_render": True,
        "manual_control": False,
        # "vehicle_config": {
        #     "spawn_lane_index": (FirstPGBlock.NODE_2, FirstPGBlock.NODE_3, 2),
        #     "show_dest_mark": True,
        #     "show_navi_mark": True,
        #     # Whether to draw a line from current vehicle position to the designation point
        #     "show_line_to_dest":True,
        #     # Whether to draw a line from current vehicle position to the next navigation point
        #     "show_line_to_navi_mark":True,
        #     # Whether to draw left / right arrow in the interface to denote the navigation direction
        #     "show_navigation_arrow":True,
        #     # If set to True, the vehicle will be in color green in top-down renderer or MARL setting
        #     "use_special_color":True,
        # },
        })
module = rl_module
# module = algo_loaded.get_module()
action_dist_class = module.get_inference_action_dist_cls()
obs, info = env.reset(seed=0)
total_cost = 0
try:
    for i in range(1, 10_000):
        if i % 100 == 0:
            print(f"Iteration {i}")
        fwd_ins = {"obs": torch.Tensor([obs])}
        fwd_outputs = module.forward_exploration(fwd_ins)
        # This can be either deterministic or stochastic distribution.
        action_dist = action_dist_class.from_logits(
            fwd_outputs["action_dist_inputs"]
        )
        action = action_dist.sample()[0].numpy()
        obs, reward, terminated, truncated, info = env.step(action)
        total_cost += info["cost"]
        ret = env.render(
            text={
                "cost": total_cost,
                "seed": env.current_seed,
                "reward": reward,
                "total_cost": info["total_cost"],
            }
        )
        if info["crash_vehicle"]:
            print("crash_vehicle:cost {}, reward {}".format(info["cost"], reward))
        if info["crash_object"]:
            print("crash_object:cost {}, reward {}".format(info["cost"], reward))

        if (terminated or truncated) and info["arrive_dest"]:
            total_cost = 0
            print("done_cost:{}".format(info["cost"]), "done_reward;{}".format(reward))
            print("Reset")
            env.reset(env.current_seed + 1)
finally:
    env.close()
print("Finished")

[INFO] Environment: SafeMetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector(), main_camera: MainCamera(1200, 900), dashboard: DashBoard()]
[INFO] Render Mode: onscreen
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: CocoaGraphicsPipe
[INFO] Start Scenario Index: 0, Num Scenarios : 100


Iteration 100
Iteration 200
Iteration 300
Iteration 400
Iteration 500
Iteration 600
Iteration 700
Iteration 800
Iteration 900
Iteration 1000
Iteration 1100
Iteration 1200
Iteration 1300
Iteration 1400
Iteration 1500
Iteration 1600
Iteration 1700
Iteration 1800
Iteration 1900
Iteration 2000
Iteration 2100
Iteration 2200
Iteration 2300
Iteration 2400
Iteration 2500
Iteration 2600
Iteration 2700
Iteration 2800
Iteration 2900
Iteration 3000
Iteration 3100
Iteration 3200
Iteration 3300
Iteration 3400
Iteration 3500
Iteration 3600
Iteration 3700
Iteration 3800
Iteration 3900
Iteration 4000
Iteration 4100
Iteration 4200


SystemExit: 

/Users/jameswalker/miniconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
